# NB03 — Baseline LSTM et Entraînement
> **Phase C & D** : Hochreiter & Schmidhuber 1997, Bengio et al. 2003.


In [ ]:
# ==============================================================================
# Bootstrap : détection du répertoire racine (local ET Google Colab)
# ==============================================================================
import sys
from pathlib import Path

def _find_root():
    """Walk up from cwd looking for utils.py (also checks <p>/project1/utils.py)."""
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for p in candidates:
        if (p / "utils.py").exists():
            return p
        if (p / "project1" / "utils.py").exists():
            # utils.py lives at repo root, not inside project1
            return p
    return None

ROOT_DIR = _find_root()

if ROOT_DIR is None:
    # ── Colab fallback: clone the repo ──────────────────────────────────────
    if Path("/content").exists():
        import subprocess
        clone_target = Path("/content/Recherche_AI")
        if not clone_target.exists():
            subprocess.run(
                ["git", "clone", "https://github.com/AhmadBelhaj4/Recherche_AI.git",
                 str(clone_target)],
                check=True,
            )
        ROOT_DIR = _find_root()

if ROOT_DIR is None:
    raise FileNotFoundError(
        "Could not locate utils.py. "
        "Set AI_PROJECT_DIR or run from inside the Recherche_AI repo."
    )

sys.path.insert(0, str(ROOT_DIR))
from utils import get_paths
PATHS = get_paths()
print(f"ROOT_DIR  : {ROOT_DIR}")
print(f"PATHS     : {PATHS}")


In [ ]:
# ==============================================================================
# Cellule 1 : Paramètres (Settings) et reproductibilité
# ==============================================================================
import sys
import math
from pathlib import Path
import pandas as pd
import torch
import torch.nn as nn

from utils import (set_seed, get_device, AminoAcidTokenizer,
                   SequenceDataset, collate_fn, make_loader, shift,
                   compute_nats_and_perplexity)
from models import LSTMLM

SEED = 42
set_seed(SEED)
device = get_device()

BATCH_SIZE = 64
EMBED_DIM = 128
HIDDEN_DIM = 256
NUM_LAYERS = 1
LR = 1e-3
WEIGHT_DECAY = 0.01
EPOCHS = 10
GRAD_CLIP = 1.0

print(f"Device: {device} | Seed: {SEED} | Epochs: {EPOCHS} | LR: {LR}")


In [ ]:
# ==============================================================================
# Cellule 2 : Architecture LSTM avec Weight Tying
# ==============================================================================
# LSTMLM est défini dans models.py (importé dans la cellule 1)

tokenizer = AminoAcidTokenizer()
model = LSTMLM(tokenizer.vocab_size, EMBED_DIM, HIDDEN_DIM, NUM_LAYERS).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Modèle LSTM instancié : {n_params:,} paramètres entraînables.")


In [ ]:
# ==============================================================================
# Cellule 3 : Données, Optimiseur AdamW et Fonction d'évaluation (nats/token)
# ==============================================================================
train_path = PATHS["raw"].parent / "train.csv"
val_path = PATHS["raw"].parent / "val.csv"
if not train_path.exists():
    train_path = PATHS["raw"].parent / "toy" / "train.csv"
    val_path = PATHS["raw"].parent / "toy" / "val.csv"

train_seqs = pd.read_csv(train_path)["sequence"].tolist()
val_seqs = pd.read_csv(val_path)["sequence"].tolist()

train_loader = make_loader(train_seqs, tokenizer, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = make_loader(val_seqs,   tokenizer, batch_size=BATCH_SIZE, shuffle=False)

criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_id, reduction='sum')
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    with torch.no_grad():
        for batch in loader:
            inputs, targets = shift(batch)
            inputs, targets = inputs.to(device), targets.to(device)
            logits, _ = model(inputs)
            loss = criterion(logits.view(-1, tokenizer.vocab_size), targets.view(-1))
            non_pad = (targets != tokenizer.pad_id).sum().item()
            total_loss += loss.item()
            total_tokens += non_pad
    return compute_nats_and_perplexity(total_loss, total_tokens)

print(f"Données prêtes : Train={len(train_seqs):,} | Val={len(val_seqs):,}")


In [ ]:
# ==============================================================================
# Cellule 4 : Boucle d'entraînement (10 epochs) et sauvegarde du meilleur modèle
# ==============================================================================
import matplotlib.pyplot as plt

history = []
best_val_nats = float('inf')
models_dir = PATHS["models"]
models_dir.mkdir(parents=True, exist_ok=True)
best_model_path = models_dir / "lstm_best.pt"

print("Début de l'entraînement LSTM...")
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_train_loss = 0.0
    total_train_tokens = 0

    for batch in train_loader:
        inputs, targets = shift(batch)
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        logits, _ = model(inputs)
        loss = criterion(logits.view(-1, tokenizer.vocab_size), targets.view(-1))
        non_pad = (targets != tokenizer.pad_id).sum().item()

        (loss / max(non_pad, 1)).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        total_train_loss += loss.item()
        total_train_tokens += non_pad

    train_nats, train_ppl = compute_nats_and_perplexity(total_train_loss, total_train_tokens)
    val_nats, val_ppl = evaluate(model, val_loader)
    history.append((epoch, train_nats, val_nats))

    if val_nats < best_val_nats:
        best_val_nats = val_nats
        torch.save(model.state_dict(), best_model_path)
        star = "★"
    else:
        star = ""

    print(f"Epoch {epoch:2d}/{EPOCHS} | Train nats: {train_nats:.3f} (ppl: {train_ppl:.1f}) | Val nats: {val_nats:.3f} (ppl: {val_ppl:.1f}) {star}")

print(f"Entraînement terminé. Meilleur val nats: {best_val_nats:.3f} sauvegardé dans {best_model_path}")

# Affichage des courbes
epochs_range = [h[0] for h in history]
plt.figure(figsize=(7, 4))
plt.plot(epochs_range, [h[1] for h in history], label="Train nats/token")
plt.plot(epochs_range, [h[2] for h in history], label="Val nats/token")
plt.xlabel("Epoch")
plt.ylabel("nats / token")
plt.title("Courbe d'apprentissage LSTM")
plt.legend()
plt.tight_layout()
plt.show()
